In [ ]:
# 1. Setup and Project Configuration
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt

project_id = 'your-gcp-project-id'
client = bigquery.Client(project=project_id)

# 2. Model Training (ARIMA+ for Time-Series Forecasting)
# This SQL creates a model directly in BigQuery to predict the next 24 hours of temperature
model_name = "iot_analytics_ds.sensor_forecast_model"
train_query = f"""
CREATE OR REPLACE MODEL `{model_name}`
OPTIONS(
  model_type='ARIMA_PLUS',
  time_series_timestamp_col='timestamp',
  time_series_data_col='temperature',
  time_series_id_col='device_id',
  horizon=24,
  auto_arima=TRUE
) AS
SELECT 
    timestamp, 
    device_id, 
    temperature 
FROM `your-dataset.iot_telemetry_table`
WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
"""
client.query(train_query).result()
print("Model training complete.")

# 3. Generating Forecasts
forecast_query = f"""
SELECT
 *
FROM
 ML.FORECAST(MODEL `{model_name}`,
             STRUCT(24 AS horizon, 0.95 AS confidence_level))
"""
forecast_df = client.query(forecast_query).to_dataframe()

# 4. Visualizing Predicted vs. Actual
plt.figure(figsize=(12, 6))
plt.plot(forecast_df['forecast_timestamp'], forecast_df['forecast_value'], label='Predicted Temp', color='orange')
plt.fill_between(forecast_df['forecast_timestamp'], 
                 forecast_df['prediction_interval_lower_bound'], 
                 forecast_df['prediction_interval_upper_bound'], 
                 color='orange', alpha=0.2, label='95% Confidence')
plt.title('24-Hour Temperature Forecast for IoT Devices')
plt.legend()
plt.show()